# Options from Zero · Episode 14: The Black-Scholes equation from hedging

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. The hedged book · 4. The proof in numbers: one day, hedged and unhedged · 5. No risk, so the risk-free return · 6. Solve it on a grid · 7. This episode's question · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
mu = 0.08   # the share's expected return (real world): it will drop out

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 14,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

In [ ]:
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": r * 100, "vol_pct": vol * 100}

Part 2's additions: Greeks, simulation, hedging, implied volatility, Black and Bachelier (`episodes/_shared/options_part2.py`):

In [ ]:
# Part 2 additions to options.py: the Greeks, simulation, a hedging simulator, implied volatility, and Black's and
# Bachelier's formulas on a forward (for FX and interest-rate options). Written out by hand and copied verbatim into
# each Part 2 notebook; the notebooks check them against QuantLib. Needs options.py (norm_cdf, black_scholes),
# math and numpy as np.

def norm_pdf(x):
    """The bell curve's height at x (standard normal density)."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def norm_cdf_v(x):
    """norm_cdf for a whole numpy array at once."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x, dtype=float) / math.sqrt(2)))

def bs_greeks(s, k, t, r, vol):
    """Price and Greeks of a European call on a share with no dividends.
    delta per $1 of share price, gamma per $1 squared, vega per 1.00 of volatility (100 vol points),
    theta per year (negative: value lost as time passes), rho per 1.00 of interest rate."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    disc = math.exp(-r * t)
    return {"price": s * norm_cdf(d1) - k * disc * norm_cdf(d2),
            "delta": norm_cdf(d1),
            "gamma": norm_pdf(d1) / (s * vol * math.sqrt(t)),
            "vega": s * norm_pdf(d1) * math.sqrt(t),
            "theta": -s * norm_pdf(d1) * vol / (2 * math.sqrt(t)) - r * k * disc * norm_cdf(d2),
            "rho": k * t * disc * norm_cdf(d2)}

def call_delta(s, k, t, r, vol):
    """Black-Scholes delta N(d1) for an array of share prices."""
    d1 = (np.log(np.asarray(s, dtype=float) / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    return norm_cdf_v(d1)

def simulate_paths(s, t, drift, vol, steps, n_paths, rng):
    """Share price paths: shape (n_paths, steps + 1), starting at s, lognormal steps with the given drift."""
    dt = t / steps
    z = rng.standard_normal((n_paths, steps))
    log_steps = (drift - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * z
    return s * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_steps, axis=1)], axis=1))

def monte_carlo_call(s, k, t, r, vol, n, rng):
    """Price = the average discounted payoff over n risk-neutral outcomes, with its standard error."""
    z = rng.standard_normal(n)
    s_t = s * np.exp((r - vol ** 2 / 2) * t + vol * math.sqrt(t) * z)
    pv = math.exp(-r * t) * np.maximum(s_t - k, 0.0)
    return float(pv.mean()), float(pv.std(ddof=1) / math.sqrt(n))

def hedge_short_call(s, k, t, r, vol_implied, vol_realised, steps, n_paths, rng, hedge=True):
    """Sell the call at vol_implied and, if hedge, rebalance to delta shares (computed at vol_implied) `steps` times.
    The share moves with vol_realised and grows at r. Cash earns r. Returns each path's P&L at expiry in today's dollars."""
    dt = t / steps
    price = s * np.ones(n_paths)
    held = call_delta(price, k, t, r, vol_implied) if hedge else np.zeros(n_paths)
    cash = black_scholes(s, k, t, r, vol_implied) - held * price      # premium received, shares bought
    for i in range(1, steps + 1):
        price = price * np.exp((r - vol_realised ** 2 / 2) * dt + vol_realised * math.sqrt(dt) * rng.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        if hedge and i < steps:
            new = call_delta(price, k, t - i * dt, r, vol_implied)
            cash = cash - (new - held) * price                        # buy (or sell) the difference
            held = new
    pnl = cash + held * price - np.maximum(price - k, 0.0)           # sell the shares, pay the option's payoff
    return pnl * math.exp(-r * t)

def implied_vol(price, s, k, t, r, lo=0.0001, hi=3.0, tol=1e-12):
    """Run Black-Scholes backwards by bisection: halve [lo, hi] until the price fits. Returns (vol, halvings)."""
    n = 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if black_scholes(s, k, t, r, mid) < price:
            lo = mid
        else:
            hi = mid
        n += 1
    return (lo + hi) / 2, n

def black76(forward, strike, t, vol, df, call=True):
    """Black's formula on a forward: df x [F N(d1) - K N(d2)] for a call. FX options (Garman-Kohlhagen) and caplets."""
    sd = vol * math.sqrt(t)
    d1 = (math.log(forward / strike) + sd ** 2 / 2) / sd
    d2 = d1 - sd
    if call:
        return df * (forward * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - forward * norm_cdf(-d1))

def bachelier(forward, strike, t, normal_vol, df, call=True):
    """Bachelier's formula: the forward moves by normal_vol x sqrt(t) in absolute terms (it can go below zero)."""
    sd = normal_vol * math.sqrt(t)
    d = (forward - strike) / sd
    if call:
        return df * ((forward - strike) * norm_cdf(d) + sd * norm_pdf(d))
    return df * ((strike - forward) * norm_cdf(-d) + sd * norm_pdf(d))

## 2. Last episode's question

A 40% volatility share with an 8% expected return: its median grows at $8\% - 0.40^2/2 = 0\%$.

In [ ]:
out["answer"] = {"vol_pct": 40, "mu_pct": 8, "drag_pct": 0.40 ** 2 / 2 * 100, "median_pct": round((0.08 - 0.40 ** 2 / 2) * 100, 10) + 0.0}
out["answer"]

## 3. The hedged book

Own the option (value $V$) and sell $\Delta$ shares short: $\Pi = V - \Delta S$. Itô's lemma (Episode 13) for $V(S,t)$:

$$dV = \frac{\partial V}{\partial t}dt + \frac{\partial V}{\partial S}dS + \frac12\frac{\partial^2V}{\partial S^2}\sigma^2S^2\,dt$$

so $d\Pi = dV - \Delta\,dS$. Choose $\Delta = \partial V/\partial S$ and every $dS$ term, the only random part, cancels:

$$d\Pi = \Big(\frac{\partial V}{\partial t} + \frac12\sigma^2S^2\frac{\partial^2V}{\partial S^2}\Big)dt$$

## 4. The proof in numbers: one day, hedged and unhedged

20,000 scenarios for tomorrow's share price (expected return 8%, volatility 20%). Unhedged: the change in the call's value. Hedged: the same, minus delta times the share's move, minus a day's interest on the cash the hedge frees up.

In [ ]:
g = bs_greeks(S, K, T, r, vol)
rng = np.random.default_rng(14)
dt = 1 / 365
S1 = S * np.exp((mu - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * rng.standard_normal(20_000))
C1 = np.array([black_scholes(s, K, T - dt, r, vol) for s in S1])
unhedged = C1 - g["price"]
hedged = unhedged - g["delta"] * (S1 - S) - r * (g["price"] - g["delta"] * S) * dt
bins = np.arange(-2.0, 2.0001, 0.05)
def shares(x):
    c, _ = np.histogram(np.clip(x, -1.999, 1.999), bins=bins)
    return (c / len(x)).round(5).tolist()
centres = ((bins[:-1] + bins[1:]) / 2).round(3).tolist()
out["one_day"] = {"n": len(S1), "unhedged_sd": float(unhedged.std()), "hedged_sd": float(hedged.std()),
                  "unhedged_mean": float(unhedged.mean()), "hedged_mean": float(hedged.mean()),
                  "risk_cut": float(1 - hedged.std() / unhedged.std()), "ratio": float(hedged.std() / unhedged.std()),
                  "hist_x": centres, "hist_unhedged": shares(unhedged), "hist_hedged": shares(hedged), "bar_width": 0.05,
                  "ticks": [{"x": x, "label": f"{'+' if x > 0 else ('−' if x < 0 else '')}${abs(x):.0f}"} for x in (-2, -1, 0, 1, 2)]}
assert out["one_day"]["risk_cut"] > 0.95
{k: v for k, v in out["one_day"].items() if not isinstance(v, list)}

## 5. No risk, so the risk-free return

A portfolio with no risk must earn the risk-free rate, or there's an arbitrage (Episode 3): $d\Pi = r\,\Pi\,dt = r\,(V - \Delta S)\,dt$. Setting the two expressions for $d\Pi$ equal:

$$\frac{\partial V}{\partial t} + \frac12\sigma^2S^2\frac{\partial^2V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV = 0$$

This is the **Black-Scholes equation**, a partial differential equation (PDE). The expected return $\mu$ appears nowhere: it went out with the $dS$ terms. Check: the formula's price satisfies it (derivatives by nudging).

In [ ]:
h, ht = 0.01, 1e-5
V = lambda s, t: black_scholes(s, K, t, r, vol)       # t = time to expiry
V_t = -(V(S, T + ht) - V(S, T - ht)) / (2 * ht)        # calendar time runs the other way
V_S = (V(S + h, T) - V(S - h, T)) / (2 * h)
V_SS = (V(S + h, T) - 2 * V(S, T) + V(S - h, T)) / h ** 2
residual = V_t + 0.5 * vol ** 2 * S ** 2 * V_SS + r * S * V_S - r * V(S, T)
out["pde"] = {"V_t": V_t, "V_S": V_S, "V_SS": V_SS, "residual": residual}
assert abs(residual) < 1e-4
out["pde"]

## 6. Solve it on a grid

The equation can be solved without the formula: put the share price on a grid of points, start from the payoff at expiry, and step back to today, each step replacing the derivatives with differences between neighbouring points (**finite differences**). Our explicit scheme below uses a grid from \$0 to \$250; QuantLib's `FdBlackScholesVanillaEngine` uses its own grid (in log price) with the same number of points in price and in time. Both converge to the formula's \$7.39 as the grid gets finer.

In [ ]:
def fd_explicit(points, s_max=250.0):
    '''Explicit finite differences for a call: step back from the payoff, one small time step at a time.'''
    ds = s_max / points
    grid = np.linspace(0.0, s_max, points + 1)
    dt_max = ds ** 2 / (vol ** 2 * s_max ** 2)                 # stability limit of the explicit scheme
    n_t = math.ceil(T / (0.9 * dt_max))
    dt = T / n_t
    v = np.maximum(grid - K, 0.0)
    for j in range(1, n_t + 1):
        tau = j * dt                                            # time to expiry after this step
        i = slice(1, -1)
        gamma_ = (v[2:] - 2 * v[1:-1] + v[:-2]) / ds ** 2
        delta_ = (v[2:] - v[:-2]) / (2 * ds)
        v[i] = v[i] + dt * (0.5 * vol ** 2 * grid[i] ** 2 * gamma_ + r * grid[i] * delta_ - r * v[i])
        v[0], v[-1] = 0.0, s_max - K * math.exp(-r * tau)
    return float(np.interp(S, grid, v)), n_t

today = ql.Date(29, 9, 2026)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
process = ql.BlackScholesMertonProcess(
    ql.QuoteHandle(ql.SimpleQuote(S)), flat(0.0), flat(r),
    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), vol, dc)))
expiry = today + 365   # exactly one year on ACT/365F
opt = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, K), ql.EuropeanExercise(expiry))
formula = black_scholes(S, K, T, r, vol)
rows = []
for n in (25, 100, 400):
    ours, n_t = fd_explicit(n)
    opt.setPricingEngine(ql.FdBlackScholesVanillaEngine(process, n, n))
    rows.append({"points": n, "ours": ours, "time_steps": n_t, "quantlib": opt.NPV(), "formula": formula})
assert abs(rows[-1]["ours"] - formula) < 0.005 and abs(rows[-1]["quantlib"] - formula) < 0.005
out["fd"] = {"rows": rows, "formula": formula, "q25": rows[0]["quantlib"], "q100": rows[1]["quantlib"], "q400": rows[2]["quantlib"],
             "o25": rows[0]["ours"], "o400": rows[2]["ours"]}
pd.DataFrame(rows)

## 7. This episode's question

Why isn't the hedged spread exactly zero? Because we set the hedge once and left it for a whole day. The argument above needs the hedge adjusted continuously. Hedge over a shorter interval and the leftover risk shrinks, roughly with the square root of the interval:

In [ ]:
ratios = {}
for label, dt_ in (("day", 1 / 365), ("hour", 1 / (365 * 24))):
    s1 = S * np.exp((mu - vol ** 2 / 2) * dt_ + vol * math.sqrt(dt_) * rng.standard_normal(20_000))
    un = np.array([black_scholes(s, K, T - dt_, r, vol) for s in s1]) - g["price"]
    he = un - g["delta"] * (s1 - S) - r * (g["price"] - g["delta"] * S) * dt_
    ratios[label] = float(he.std() / un.std())
out["question"] = {"left_day": ratios["day"], "left_hour": ratios["hour"]}
out["question"]

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")